# CIFAR-10 Image Classification Using a CNN

This project builds a Convolutional Neural Network (CNN) to classify images from the CIFAR-10 dataset into 10 categories.

**Dataset:** CIFAR-10 (32 × 32 colour images)  
**Framework:** TensorFlow / Keras  
**Model type:** Convolutional Neural Network

The notebook covers dataset loading, visualisation, preprocessing, model creation, training, evaluation, and sample predictions.

> The notebook downloads CIFAR-10 through Keras the first time it runs. An internet connection may be needed for that download.

## 1. Import libraries
The imports below are used for loading the dataset, building and training the CNN, plotting results, and evaluating predictions.

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Set random seeds to make runs more repeatable
tf.keras.utils.set_random_seed(42)

## 2. Load the CIFAR-10 dataset
Keras provides the CIFAR-10 dataset as training and test sets. The test set is kept separate and is used only for the final evaluation.

In [ ]:
(X_train, y_train), (X_test, y_test) = datasets.cifar10.load_data()

print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", X_test.shape)
print("Test labels:", y_test.shape)

## 3. Explore the dataset
Display one example from each of the first ten training images to see the image format and class labels.

In [ ]:
class_names = [
    "Airplane", "Automobile", "Bird", "Cat", "Deer",
    "Dog", "Frog", "Horse", "Ship", "Truck"
]

plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i])
    plt.title(class_names[int(y_train[i][0])])
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Preprocess the images
Pixel values originally range from 0 to 255. Dividing by 255 scales them to the range 0 to 1, which is suitable for neural-network training.

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

print("Training pixel range:", X_train.min(), "to", X_train.max())
print("Test pixel range:", X_test.min(), "to", X_test.max())

## 5. Build the CNN model
The model uses convolution and max-pooling layers to learn image features. A dense layer then uses those features to classify each image into one of the ten CIFAR-10 classes.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.summary()

## 6. Compile and train the model
Sparse categorical cross-entropy is used because the labels are integer class IDs. Ten percent of the training data is used for validation. The separate test set is not used during training.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=15,
    validation_split=0.1,
    batch_size=64
)

## 7. View training results
The plots show how accuracy and loss changed over the training epochs. Validation metrics are calculated using the held-out portion of the training data.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Model accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Model loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

### Interpreting the training curves

After running the notebook, compare the training and validation curves:

- If both accuracies improve and both losses decrease, the model is learning useful patterns.
- If training accuracy continues to rise while validation accuracy stops improving or falls, this may indicate overfitting.
- If both accuracies remain low, the model may be underfitting.

Write your interpretation based on the curves produced by your own run. Do not assume a particular outcome before running the notebook.

## 8. Evaluate on the test set
This is the final evaluation on CIFAR-10 test images, which were kept separate from training and validation.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")

## 9. Classification report and confusion matrix
The classification report gives precision, recall, and F1-score for each class. The confusion matrix shows which classes the model predicts correctly or mixes up.

In [ ]:
y_probabilities = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_probabilities, axis=1)
y_true = y_test.flatten()

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
plt.imshow(cm, cmap="Blues")
plt.title("CIFAR-10 Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(range(10), class_names, rotation=45, ha="right")
plt.yticks(range(10), class_names)
plt.colorbar(label="Number of images")
plt.tight_layout()
plt.show()

## 10. View sample predictions
The following cell displays ten test images with the model's predicted class and its confidence. The true label is also shown for comparison.

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i])

    predicted_class = int(y_pred[i])
    true_class = int(y_true[i])
    confidence = float(np.max(y_probabilities[i])) * 100

    plt.title(
        f"Predicted: {class_names[predicted_class]}\n"
        f"True: {class_names[true_class]}\n"
        f"Confidence: {confidence:.1f}%"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## 11. Save the trained model
Save the trained Keras model so it can be loaded later without training it again.

In [ ]:
model.save("cifar10_cnn_model.keras")
print("Model saved as cifar10_cnn_model.keras")

## 12. Conclusion

In this project, a CNN was built and trained to classify CIFAR-10 images into ten categories. The model's performance was examined using training and validation curves, test accuracy, a classification report, a confusion matrix, and sample predictions.

The test accuracy printed in Section 8 is the result from the particular run of this notebook. Results can vary depending on the training environment and configuration.

## Possible future improvements

- Experiment with more epochs and different batch sizes.
- Add data augmentation to improve generalisation.
- Try additional convolutional layers or regularisation.
- Compare the CNN with other model architectures.

## Requirements

- Python
- TensorFlow
- NumPy
- Matplotlib
- scikit-learn

Install the libraries if needed:

```bash
pip install tensorflow numpy matplotlib scikit-learn
```